In [22]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import re
import janitor

In [ ]:
#race = pd.read_csv("data/philly_race_ethn_deceniall_p6/DECENNIALDHC2020.P9-Data.csv", skiprows=1)
race = pd.read_csv("data/2010_p9_philly/DECENNIALSF12010.P9-Data.csv", skiprows=1)
year = 2010
race.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/2010_p9_philly/DECENNIALSF12010.P9-Data.csv'

In [ ]:
race.shape

(385, 77)

In [ ]:
race["TRACTCE"] = (
    race["Geographic Area Name"]
    .str.extract(r"Census Tract ([\d\.]+)")[0]   # extract tract number
    .str.replace(".", "", regex=False)           # remove decimal point
    .str.zfill(6)                                # pad to 6 digits
)


race['TRACTCE']

0      000001
1      000002
2      000003
3      000401
4      000402
        ...  
380    009807
381    009808
382    009809
383    009891
384       NaN
Name: TRACTCE, Length: 385, dtype: object

In [ ]:

def tract_to_tractce(geo_name):
    # extract the tract number from the string
    match = re.search(r'Census Tract ([\d.]+)', geo_name)
    if not match:
        return None
    
    tract = match.group(1)
    
    # split on decimal point
    if '.' in tract:
        integer_part, decimal_part = tract.split('.')
        # Pad decimal to 2 digits (e.g., "1" -> "01")
        decimal_part = decimal_part.ljust(2, '0')
    else:
        integer_part = tract
        decimal_part = '00'
    
    # combine and zero-pad to 6 digits
    tractce = f"{int(integer_part):04d}{decimal_part}"
    return tractce

race['TRACTCE'] = race['Geographic Area Name'].apply(tract_to_tractce).astype(str)
race['TRACTCE']

0      000100
1      000200
2      000300
3      000401
4      000402
        ...  
380    980700
381    980800
382    980900
383    989100
384      None
Name: TRACTCE, Length: 385, dtype: object

In [ ]:
# clean names
race_clean = race.clean_names()
race_clean.columns = race_clean.columns.str.replace("!!", "_", regex=False)
race_clean.columns = race_clean.columns.str.replace("__", "_", regex=False)
race_clean.columns = race_clean.columns.str.replace("_total_", "total_", regex=False)
race_clean.head()

,geography,geographic_area_name,total,total_hispanic_or_latino,total_not_hispanic_or_latino,total_not_hispanic_or_latino_population_of_one_race,total_not_hispanic_or_latino_population_of_one_race_white_alone,total_not_hispanic_or_latino_population_of_one_race_black_or_african_american_alone,total_not_hispanic_or_latino_population_of_one_race_american_indian_and_alaska_native_alone,total_not_hispanic_or_latino_population_of_one_race_asian_alone,...,total_not_hispanic_or_latino_two_or_more_races_population_of_five_races_white;_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander,total_not_hispanic_or_latino_two_or_more_races_population_of_five_races_white;_black_or_african_american;_american_indian_and_alaska_native;_asian;_some_other_race,total_not_hispanic_or_latino_two_or_more_races_population_of_five_races_white;_black_or_african_american;_american_indian_and_alaska_native;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_two_or_more_races_population_of_five_races_white;_black_or_african_american;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_two_or_more_races_population_of_five_races_white;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_two_or_more_races_population_of_five_races_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_two_or_more_races_population_of_six_races,total_not_hispanic_or_latino_two_or_more_races_population_of_six_races_white;_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,unnamed_75,tractce
0,1400000US42101000100,"Census Tract 1, Philadelphia County, Pennsylvania",3478,126,3352,3282,2890,207,5,173,...,0,0,0,0,0,0,0,0,NaN,000100
1,1400000US42101000200,"Census Tract 2, Philadelphia County, Pennsylvania",2937,79,2858,2817,665,284,3,1855,...,0,0,0,0,0,0,0,0,NaN,000200
2,1400000US42101000300,"Census Tract 3, Philadelphia County, Pennsylvania",3169,135,3034,2954,2290,324,1,328,...,0,0,0,0,0,0,0,0,NaN,000300
3,1400000US42101000401,"Census Tract 4.01, Philadelphia County, Pennsy...",2125,107,2018,1955,1049,376,8,519,...,0,0,0,0,0,0,0,0,NaN,000401
4,1400000US42101000402,"Census Tract 4.02, Philadelphia County, Pennsy...",3142,109,3033,2999,2455,173,3,356,...,0,0,0,0,0,0,0,0,NaN,000402


In [ ]:

# create variables of total and proportion of at least part Black and Asian
black_cols = [col for col in race_clean.columns if "black_or_african_american" in col.lower()]
asian_cols = [col for col in race_clean.columns if "asian" in col.lower()]

race_clean["total_black_any"] = race_clean[black_cols].sum(axis=1)
race_clean["total_asian_any"] = race_clean[asian_cols].sum(axis=1)
# use total_ for 2020 census
race_clean['prop_black'] = race_clean['total_black_any']/race_clean['total']
race_clean['prop_asian'] = race_clean['total_asian_any']/race_clean['total']
race_clean['prop_hispanic_or_latino'] = race_clean['total_hispanic_or_latino']/race_clean['total']
race_clean.head()

/var/folders/rz/h3gzrv2j09s6d_5vmn0bglw40000gn/T/ipykernel_94242/543657419.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race_clean["total_black_any"] = race_clean[black_cols].sum(axis=1)
/var/folders/rz/h3gzrv2j09s6d_5vmn0bglw40000gn/T/ipykernel_94242/543657419.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race_clean["total_asian_any"] = race_clean[asian_cols].sum(axis=1)
/var/folders/rz/h3gzrv2j09s6d_5vmn0bglw40000gn/T/ipykernel_94242/543657419.py:8: SettingWithCopyWarning: 
A value is trying 

,geography,geographic_area_name,total,total_hispanic_or_latino,total_not_hispanic_or_latino,total_not_hispanic_or_latino_population_of_one_race,total_not_hispanic_or_latino_population_of_one_race_white_alone,total_not_hispanic_or_latino_population_of_one_race_black_or_african_american_alone,total_not_hispanic_or_latino_population_of_one_race_american_indian_and_alaska_native_alone,total_not_hispanic_or_latino_population_of_one_race_asian_alone,...,total_not_hispanic_or_latino_two_or_more_races_population_of_five_races_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_two_or_more_races_population_of_six_races,total_not_hispanic_or_latino_two_or_more_races_population_of_six_races_white;_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,unnamed_75,tractce,total_black_any,total_asian_any,prop_black,prop_asian,prop_hispanic_or_latino
0,1400000US42101000100,"Census Tract 1, Philadelphia County, Pennsylvania",3478,126,3352,3282,2890,207,5,173,...,0,0,0,NaN,000100,222,436,0.063830,0.125359,0.036228
1,1400000US42101000200,"Census Tract 2, Philadelphia County, Pennsylvania",2937,79,2858,2817,665,284,3,1855,...,0,0,0,NaN,000200,302,3762,0.102826,1.280899,0.026898
2,1400000US42101000300,"Census Tract 3, Philadelphia County, Pennsylvania",3169,135,3034,2954,2290,324,1,328,...,0,0,0,NaN,000300,342,774,0.107920,0.244241,0.042600
3,1400000US42101000401,"Census Tract 4.01, Philadelphia County, Pennsy...",2125,107,2018,1955,1049,376,8,519,...,0,0,0,NaN,000401,396,1116,0.186353,0.525176,0.050353
4,1400000US42101000402,"Census Tract 4.02, Philadelphia County, Pennsy...",3142,109,3033,2999,2455,173,3,356,...,0,0,0,NaN,000402,178,766,0.056652,0.243794,0.034691


In [ ]:
race_sub = race_clean[['geography', 'geographic_area_name', 'total', 'tractce', 'prop_black', 'prop_asian', 'prop_hispanic_or_latino']]

race_sub = race_sub.rename(columns={"total": "tot_pop", 
                                    "tractce": "TRACTCE"})

race_sub.head()

,geography,geographic_area_name,tot_pop,TRACTCE,prop_black,prop_asian,prop_hispanic_or_latino
0,1400000US42101000100,"Census Tract 1, Philadelphia County, Pennsylvania",3478,000100,0.063830,0.125359,0.036228
1,1400000US42101000200,"Census Tract 2, Philadelphia County, Pennsylvania",2937,000200,0.102826,1.280899,0.026898
2,1400000US42101000300,"Census Tract 3, Philadelphia County, Pennsylvania",3169,000300,0.107920,0.244241,0.042600
3,1400000US42101000401,"Census Tract 4.01, Philadelphia County, Pennsy...",2125,000401,0.186353,0.525176,0.050353
4,1400000US42101000402,"Census Tract 4.02, Philadelphia County, Pennsy...",3142,000402,0.056652,0.243794,0.034691


In [ ]:
race_sub['prop_white_only'] = 1 - (race_sub['prop_black'] + race_sub['prop_asian'] + race_sub['prop_hispanic_or_latino'])
race_sub.head()

,geography,geographic_area_name,tot_pop,TRACTCE,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only
0,1400000US42101000100,"Census Tract 1, Philadelphia County, Pennsylvania",3478,000100,0.063830,0.125359,0.036228,0.774583
1,1400000US42101000200,"Census Tract 2, Philadelphia County, Pennsylvania",2937,000200,0.102826,1.280899,0.026898,-0.410623
2,1400000US42101000300,"Census Tract 3, Philadelphia County, Pennsylvania",3169,000300,0.107920,0.244241,0.042600,0.605238
3,1400000US42101000401,"Census Tract 4.01, Philadelphia County, Pennsy...",2125,000401,0.186353,0.525176,0.050353,0.238118
4,1400000US42101000402,"Census Tract 4.02, Philadelphia County, Pennsy...",3142,000402,0.056652,0.243794,0.034691,0.664863


In [ ]:
race_sub.dtypes

geography                   object
geographic_area_name        object
tot_pop                      int64
TRACTCE                     object
prop_black                 float64
prop_asian                 float64
prop_hispanic_or_latino    float64
prop_white_only            float64
dtype: object

In [ ]:
# census tract 980400 has no total population nor proportion? 
# census tract 
race_sub[race_sub.isnull().any(axis=1)]


,geography,geographic_area_name,tot_pop,TRACTCE,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only
377,1400000US42101980400,"Census Tract 9804, Philadelphia County, Pennsy...",0,980400,NaN,NaN,NaN,NaN
378,1400000US42101980500,"Census Tract 9805, Philadelphia County, Pennsy...",0,980500,NaN,NaN,NaN,NaN
379,1400000US42101980600,"Census Tract 9806, Philadelphia County, Pennsy...",0,980600,NaN,NaN,NaN,NaN


In [20]:
race_sub = race_sub.dropna()
race_sub[race_sub.isnull().any(axis=1)]

,geography,geographic_area_name,tot_pop,TRACTCE,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only


In [21]:

race_sub.to_csv(f"modified_data/race_ethn_cleaned_{year}.csv")